# Results for model: meta/llama-3.1-70b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
import numpy as np

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Convert to Pandas DataFrame
df = df.to_pandas()

# Feature Engineering
def top_quantile(group):
    return np.percentile(group['feature_00'], 85)

df['TOP_QUANTILE'] = df.groupby('date_id')['feature_00'].transform(top_quantile)

# Prepare data for training
X = df.drop(['responder_6', 'date_id'], axis=1)
y = df['responder_6']

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)

# Make predictions on validation set
y_pred = xgb_model.predict(X_val)